# 04 — Temporal Train / Validation / Test Split

**Source:** `outputs/rfm_with_labels.parquet`  
**Outputs:** `outputs/train.parquet`, `outputs/val.parquet`, `outputs/test.parquet`

| Split | Condition on `first_purchase_date` | Approx period |
|---|---|---|
| **train** | `< 2010-06-01`  | Dec 2009 – May 2010 |
| **val**   | `>= 2010-06-01` and `< 2010-09-01` | Jun – Aug 2010 |
| **test**  | `>= 2010-09-01` | Sep – Nov 2010 |

Features saved (7): `recency`, `frequency`, `monetary`, `aov`, `purchase_velocity`, `days_as_customer`, `unique_products`  
Target saved: `churn_label` (renamed from `churned`)

In [1]:
import pandas as pd
import numpy as np
import duckdb
import plotly.express as px
from pathlib import Path

In [2]:
RFM_PATH   = Path("../outputs/rfm_with_labels.parquet")
CLEAN_PATH = Path("../outputs/retail_clean.parquet")
TRAIN_PATH = Path("../outputs/train.parquet")
VAL_PATH   = Path("../outputs/val.parquet")
TEST_PATH  = Path("../outputs/test.parquet")

TRAIN_CUTOFF = pd.Timestamp("2010-06-01")
VAL_CUTOFF   = pd.Timestamp("2010-09-01")

FEATURE_COLS = [
    "recency", "frequency", "monetary", "aov",
    "purchase_velocity", "days_as_customer", "unique_products",
]
TARGET_COL = "churn_label"

assert RFM_PATH.exists(),   f"Run notebooks 02 & 03 first — {RFM_PATH} not found"
assert CLEAN_PATH.exists(), f"Run notebook 01 first — {CLEAN_PATH} not found"

---
## Section 1 — Why Temporal Split?

### The leakage problem with random splits

In this pipeline every customer's RFM features are computed over their **entire** transaction
history, and their churn label is derived from whether they bought in a fixed prediction window
(2010-12-01 onward). A **random split** would place a customer who first appeared in
October 2010 into both train and test by chance. The model learns patterns from customers
whose "future" behaviour is already baked into the label — and then gets evaluated on the
same population. This inflates test-set performance without reflecting anything real.

### What temporal splitting does instead

We split by **first purchase date** (earliest invoice per customer in `retail_clean.parquet`).
The model trains on the *oldest* cohort, validates on an intermediate cohort, and is tested on
the *most recent* cohort — exactly what happens in production, where the model is deployed
and then scored on customers who joined after the training cutoff.

| Split | `first_purchase_date` | Period |
|---|---|---|
| **train** | `< 2010-06-01`  | Dec 2009 – May 2010 |
| **val**   | `>= 2010-06-01` and `< 2010-09-01` | Jun 2010 – Aug 2010 |
| **test**  | `>= 2010-09-01` | Sep 2010 onward |

> The original spec used `< 2009-12-01` as the train cutoff, which yields 0 rows because
> the dataset only begins on 2009-12-01. Cutoffs were shifted forward to produce
> three non-empty, temporally-ordered splits.

---
## Section 2 — Execute Split

In [3]:
con = duckdb.connect()
con.execute(
    f"CREATE TABLE transactions AS SELECT * FROM read_parquet('{CLEAN_PATH.resolve().as_posix()}')"
)

first_purchase = con.execute("""
    SELECT
        "Customer ID"                   AS customer_id,
        CAST(MIN(InvoiceDate) AS DATE)  AS first_purchase_date
    FROM transactions
    GROUP BY "Customer ID"
""").df()

first_purchase["first_purchase_date"] = pd.to_datetime(first_purchase["first_purchase_date"])

print(f"first_purchase_date computed for {len(first_purchase):,} unique customers")
print(f"Date range : {first_purchase['first_purchase_date'].min().date()}  →  "
      f"{first_purchase['first_purchase_date'].max().date()}")

first_purchase_date computed for 5,878 unique customers
Date range : 2009-12-01  →  2011-12-09


In [4]:
rfm_raw = pd.read_parquet(RFM_PATH)
df = rfm_raw.merge(first_purchase, on="customer_id", how="inner")
df = df.rename(columns={"churned": TARGET_COL})

dropped = len(rfm_raw) - len(df)
print(f"Merged   : {len(df):,} customers")
if dropped:
    print(f"Dropped  : {dropped} customers with no matching transaction history")

print(f"\nfirst_purchase_date distribution:")
print(df["first_purchase_date"].describe())

Merged   : 2,835 customers

first_purchase_date distribution:
count                          2835
mean     2010-03-19 13:46:24.761904
min             2009-12-01 00:00:00
25%             2009-12-15 00:00:00
50%             2010-02-23 00:00:00
75%             2010-05-24 00:00:00
max             2010-11-28 00:00:00
Name: first_purchase_date, dtype: object


In [5]:
train = df[df["first_purchase_date"] <  TRAIN_CUTOFF].copy()
val   = df[(df["first_purchase_date"] >= TRAIN_CUTOFF) &
           (df["first_purchase_date"] <  VAL_CUTOFF)].copy()
test  = df[df["first_purchase_date"] >= VAL_CUTOFF].copy()

# Integrity checks — no overlap, complete coverage
all_ids = pd.concat([train["customer_id"], val["customer_id"], test["customer_id"]])
assert all_ids.nunique() == len(all_ids), "A customer appears in more than one split!"
assert len(train) + len(val) + len(test) == len(df), "Split sizes don't sum to total!"

print("Split integrity checks passed — no overlapping customers.")

Split integrity checks passed — no overlapping customers.


In [6]:
splits = {"train": train, "val": val, "test": test}
churn_rates = {}

header = f"{'Split':<8}  {'N':>6}  {'Churn%':>8}  {'First purch min':>17}  {'First purch max':>17}"
print(header)
print("-" * len(header))

for name, sp in splits.items():
    rate = sp[TARGET_COL].mean() * 100
    churn_rates[name] = rate
    print(
        f"{name:<8}  {len(sp):>6}  {rate:>7.1f}%  "
        f"{str(sp['first_purchase_date'].min().date()):>17}  "
        f"{str(sp['first_purchase_date'].max().date()):>17}"
    )

print()
max_diff = max(churn_rates.values()) - min(churn_rates.values())
if max_diff > 10:
    print(f"WARNING : churn rate spread is {max_diff:.1f}pp (> 10pp threshold).")
    print("  Consider stratified resampling if the model struggles on val/test.")
else:
    print(f"OK : churn rate spread is {max_diff:.1f}pp — splits are comparable.")

Split          N    Churn%    First purch min    First purch max
----------------------------------------------------------------
train       2162     21.9%         2009-12-01         2010-05-28
val          380     28.7%         2010-06-01         2010-08-31
test         293     27.6%         2010-09-01         2010-11-28

OK : churn rate spread is 6.8pp — splits are comparable.


In [7]:
summary = pd.DataFrame({
    "split":      list(splits.keys()),
    "n":          [len(s) for s in splits.values()],
    "churn_rate": [churn_rates[k] for k in splits.keys()],
})

SPLIT_COLORS = ["#2196F3", "#FF9800", "#4CAF50"]

fig_count = px.bar(
    summary, x="split", y="n",
    color="split",
    color_discrete_sequence=SPLIT_COLORS,
    text="n",
    title="Customer count per split",
    labels={"split": "Split", "n": "Customers"},
)
fig_count.update_traces(textposition="outside")
fig_count.update_layout(showlegend=False)
fig_count.show()

fig_churn = px.bar(
    summary, x="split", y="churn_rate",
    color="split",
    color_discrete_sequence=SPLIT_COLORS,
    text=summary["churn_rate"].map(lambda p: f"{p:.1f}%"),
    title="Churn rate per split — should be roughly similar across splits",
    labels={"split": "Split", "churn_rate": "Churn rate (%)"},
)
fig_churn.update_traces(textposition="outside")
fig_churn.update_layout(showlegend=False, yaxis_title="Churn rate (%)")
fig_churn.show()

---
## Section 3 — Save

Each parquet contains: `customer_id` · 7 feature columns · `churn_label` · `segment_label` = **10 columns**.  
`first_purchase_date` is excluded — it is a split key, not a model feature (would be leakage).

In [8]:
SAVE_COLS = ["customer_id"] + FEATURE_COLS + [TARGET_COL, "segment_label"]

for name, sp, path in [
    ("train", train, TRAIN_PATH),
    ("val",   val,   VAL_PATH),
    ("test",  test,  TEST_PATH),
]:
    sp[SAVE_COLS].to_parquet(path, engine="pyarrow", index=False)
    print(f"Saved {name:<5} → {path.name}  "
          f"({len(sp):,} rows × {len(SAVE_COLS)} cols)")

Saved train → train.parquet  (2,162 rows × 10 cols)
Saved val   → val.parquet  (380 rows × 10 cols)
Saved test  → test.parquet  (293 rows × 10 cols)


In [9]:
print("Verifying saved files...")
for name, path in [("train", TRAIN_PATH), ("val", VAL_PATH), ("test", TEST_PATH)]:
    chk = pd.read_parquet(path)
    assert set(chk.columns) == set(SAVE_COLS),        f"{name}: unexpected columns {chk.columns.tolist()}"
    assert chk["customer_id"].is_unique,               f"{name}: duplicate customer_id"
    assert set(chk[TARGET_COL].unique()) <= {0, 1},   f"{name}: bad target values"
    assert not chk[FEATURE_COLS].isna().any().any(),  f"{name}: NaN in features"

print("All assertions passed.")
print()
print(f"{'Split':<8}  {'Rows':>5}  {'Cols':>4}  {'Churn%':>7}")
for name, path in [("train", TRAIN_PATH), ("val", VAL_PATH), ("test", TEST_PATH)]:
    chk = pd.read_parquet(path)
    rate = chk[TARGET_COL].mean() * 100
    print(f"{name:<8}  {len(chk):>5}  {chk.shape[1]:>4}  {rate:>6.1f}%")
print()
print(f"Total  :  {sum(len(pd.read_parquet(p)) for p in [TRAIN_PATH, VAL_PATH, TEST_PATH]):,} customers")
print(f"Columns: {SAVE_COLS}")

Verifying saved files...
All assertions passed.

Split      Rows  Cols   Churn%
train      2162    10    21.9%
val         380    10    28.7%
test        293    10    27.6%

Total  :  2,835 customers
Columns: ['customer_id', 'recency', 'frequency', 'monetary', 'aov', 'purchase_velocity', 'days_as_customer', 'unique_products', 'churn_label', 'segment_label']
